# 01 評価の落とし穴 — その数字は、何を測っているのか

> 🔒 すべて**合成データ**です。⏱ 所要時間: 説明 15分 ＋ 実行 約15秒

## ゴール
- **評価点の予測誤差（RMSE）**が良くなっても、**順位**が良くなるとは限らないことを確かめる。
- 評価の**やり方**（候補の選び方）で、**数字も、手法の差も変わる**ことを知る。
- **偏りが強いほど**、観測テストの順位が、本当の良さの順位とずれることを数字で見る。
- 世界（シード）を変えた**10回の平均と誤差**で、基準を比べる。

## 前のNotebookとのつながり
00 で、**観測テストと本当の好みで、人気順が逆転**することを見ました。
ここでは、評価の**「どこを見るか」**を変えると、何が起きるかを、系統的に調べます。

## キーワード
- **RMSE**: 評価点の予測誤差（二乗平均の平方根）。**小さいほど、点をよく当てる**。
- **サンプリング評価**: 全商品を候補にせず、「正解＋ランダムに選んだ100個」だけを候補にして、順位を測る方法。計算が速いが、**数字が変わる**。
- **順位相関**: 2つの順位が、どれだけ似ているか（1＝同じ、0＝無関係）。
- **標準誤差**: 平均の誤差の目安（sd ÷ √回数）。**差がこの2倍より小さいと、はっきりした差とは言えない**。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import run_phase1

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]

e1 = run_phase1.e1()
e1.set_index("手法")

## E1. RMSE（点の予測）と、順位は、別物

**何をしているか**: 評価点を予測する4つの簡単なモデルを比べます。
- **全体平均**: 全員・全商品に、平均（3.6）を予測。
- **ユーザー平均**: ユーザーごとの平均（甘辛）を、全商品に予測。
- **商品平均**: 商品ごとの平均を予測。
- **ユーザー＋商品のバイアス**: 全体平均＋ユーザーの甘辛＋商品の質。

それぞれの**RMSE**（観測テストと、全部の組の本当の好み）と、予測した点の高い順に薦めたときの**順位の指標**を見ます。

**出力の見方**:
- **ユーザー平均**は、RMSEが 1.074 → **0.986** と、**改善**します。ところが、**順位の指標は、全体平均と全く同じ**（NDCG 0.0141、本当のprecision 0.262）。ユーザー平均は、**1人の全商品に同じ点**を予測するので、**商品の差がなく**、順位が付かないためです。**RMSEが良くなっても、順位は全く良くならない**例です。
- **ユーザー＋商品のバイアス**は、RMSEが**最も小さい**（0.878。全体平均より18%改善）。順位の指標も、観測NDCG 0.050、本当のprecision 0.654と、改善します。ただし、観測NDCGは、**人気順（0.198）の4分の1**。**点をよく当てるモデルが、順位でも最強とは限りません**。
- **人気順**は、評価点の予測では**ありません**（RMSEは測れない）。それでも、観測テストの順位では**最強**。
- RMSEは、**観測テスト**（評価された組だけ）と**全部の組**（本当の好み）で、**ほぼ同じ**（差は0.02以内）でした。この設定では、**観測で測ったRMSEは、全体の誤差を、それほど歪めていません**。（歪みが大きいのは、**順位の指標**のほうです。）
- 結論: **RMSEは、点の予測の良さであって、順位の良さではない**。推薦では、欲しいのは**順位**。

In [ ]:
e2 = run_phase1.e2()
display(e2.set_index("手法"))
fig, ax = plt.subplots(figsize=(8, 3.6))
x = np.arange(len(e2)); w = 0.27
ax.bar(x - w, e2["全商品 NDCG"], w, label="全商品を候補にする")
ax.bar(x, e2["サンプリング(一様) NDCG"], w, label="正解＋ランダム100個を候補にする")
ax.bar(x + w, e2["サンプリング(人気比例) NDCG"], w, label="正解＋人気に比例して選んだ100個")
ax.set_xticks(x); ax.set_xticklabels(e2["手法"]); ax.set(ylabel="観測NDCG@10", title="評価の候補の選び方で、数字が変わる"); ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")
fig.tight_layout()

## E2. 候補の選び方で、数字も、差も変わる

**何をしているか**: 同じ手法を、3通りの方法で評価します。
1. **全商品**を候補にして、上位10件を決める（これまでの評価）。
2. 各ユーザーについて、「**テストで高く評価した商品**＋**ランダムに選んだ100個**」だけを候補にする（**サンプリング評価**。速いので、よく使われる）。
3. 同じだが、100個を**人気に比例して**選ぶ。

**出力の見方**:
- **数字が、大きく上がります**: ランダムのNDCGが、全商品の**0.014**から、サンプリングで**0.062**（約4倍）に。**候補が少ない（正解＋100個）ので、でたらめでも当たりやすい**のです。**サンプリング評価の数字を、全商品の数字と比べてはいけません**。
- **手法の差も、変わります**: 人気順÷上限（NDCG）の比は、全商品で**約4.1倍**、一様サンプリングで**約2.2倍**、人気比例サンプリングで**約1.2倍**。**ほぼ同じ**（0.21 対 0.17）にまで縮みます。どの方法で候補を選ぶかで、「**人気順が上限よりどれだけ強いか**」の見え方が変わります。
- **順位は同じ**（人気順 ＞ 上限 ＞ 平均評価順 ＞ ランダム）ですが、**差の大きさは評価の方法次第**。
- 教訓: **評価の方法（候補の選び方）を、必ず明記する**。**別の方法で測った数字を、比べない**。

In [ ]:
e3 = run_phase1.e3()
display(e3.set_index("人気の効き pop_power"))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(e3["人気の効き pop_power"], e3["順位相関（観測 と 本当）"], "o-"); ax[0].set(title="観測テストの順位と、本当の順位の相関", xlabel="人気の効き", ylim=(0, 1.05)); ax[0].grid(alpha=0.3)
ax[1].plot(e3["人気の効き pop_power"], e3["観測NDCG: 人気順÷上限"], "o-", color="C1"); ax[1].axhline(1, color="gray", ls="--", lw=1)
ax[1].set(title="観測NDCG: 人気順 ÷ 上限（1より大きいと、人気順が上）", xlabel="人気の効き"); ax[1].grid(alpha=0.3)
fig.tight_layout()

## E3. 偏りが強いほど、観測テストの順位は、本当の順位とずれる

**何をしているか**: データを作るときの**人気の効き**（`pop_power`。大きいほど、人気の商品ばかりが評価される）を変えて、**8つの方法**（上限を除く7つ）について、
- **観測テストでの順位**（NDCG@10）
- **本当の好みでの順位**（precision@10）

の**順位相関**（1＝同じ順位、0＝無関係）を調べます（5つの世界の平均）。

**出力の見方**:
- **偏りなし（0）**: 順位相関は**0.92**。観測テストの順位は、本当の順位を、**よく再現**します。
- **偏りが出る（0.25以上）**: 相関は**0.6〜0.7に低下**し、偏りが強い（1.5）と**0.58**。**観測テストの順位は、本当の順位と、かなり食い違います**。
- 右図の**人気順÷上限**は、偏りが強いほど急に上がります（0.52 → 3.07 → 5.27）。**0.25〜0.5の間で逆転**（1を超える）。
- 結論: 観測データが偏っているほど、**観測テストの順位は、信用できなくなる**。現実のデータで、**偏りがどれだけあるか**は分からないので、**観測テストの数字を鵜呑みにしない**。

In [ ]:
e4 = run_phase1.e4()
e4.set_index("手法")

## E4. 世界を変えた10回の平均と誤差

**何をしているか**: 世界（シード）を変えた**10回**の、基準の結果を、平均と**標準誤差**（±）で比べます。1つの世界だけでは、運に左右されるためです。

**出力の見方**:
- 傾向は、**10回の世界で一貫**しています。**観測NDCG**: 人気順 **0.168（±0.008）**、上限 **0.057（±0.003）**。標準誤差は、差（0.11）に比べて、ずっと小さいので、**人気順が上限を上回るのは、誤差ではありません**。
- **本当のprecision**: 上限 **0.973** ＞ バイアスモデル 0.618 ≒ 平均評価順 0.616 ＞ 人気順（高評価）0.424 ＞ 人気順 0.305 ＞ ランダム 0.230。**バイアスモデルと平均評価順は、区別できません**（差0.002、標準誤差0.015〜0.019）。**観測NDCGでは、バイアスモデルのほうが良い**（0.059 対 0.040。差0.019は、差の標準誤差（約0.006）の約3倍）。
  - ユーザーの甘辛は、**1人の全商品に同じ値を足すだけ**なので、順位には影響しません。2つの違いは、**商品の質の推定の細かさ**だけ。**観測テストの差が、本当の良さの差とは限りません**。
- **全体平均・ユーザー平均・ランダム**は、順位がつかない（同点をランダムに崩す）ので、**同じ値**（観測NDCG 0.0138、本当のprecision 0.230）。
- 評価したユーザー数は、各世界で約720人。**少ないユーザー数が、評価の誤差の原因**になります。

## 試してみよう（演習）
1. **ユーザー平均の順位を確かめる**: E1の表の「ユーザー平均」が、「全体平均」と順位の指標が同じになる理由を、`user_mean_pred` を見て考えてください。次に、`bias_pred` の `reg_user` を `0` と `100` に変えて、RMSEと順位の指標がどう変わるか見てください。
2. **候補の数を変える**: `run_phase1.e2` の `evaluate_observed_sampled(S, val, ex, 100)` の `100` を `10` と `1000` にしてください。見るポイントは、ランダムの数字と、人気順÷上限の比です。分かるのは、候補が少ないほど数字が上がり、差が縮むことです。
3. **偏りの境目を探す**: `run_phase1.e3` の `powers` を `(0.3, 0.4, 0.5, 0.6)` にして、人気順÷上限が1を超える境目を、細かく探してください。
4. **世界を増やす**: `SEEDS` を10個から30個にして（`range(30)`）、`e4` の標準誤差が小さくなるか見てください。

## まとめ
- **RMSEが良くなっても、順位が良くなるとは限らない**。ユーザー平均は、RMSEが約8%（1.074→0.986）改善しても、順位は全体平均と**全く同じ**。**推薦では、順位を測る**。
- **評価の方法（候補の選び方）**で、数字も差も変わる。サンプリング評価では、ランダムでも約4倍の数字が出る。**方法を明記し、別の方法の数字を比べない**。
- **偏りが強いほど**、観測テストの順位は、本当の順位とずれる（順位相関 0.92 → 0.58〜0.68）。観測テストを**鵜呑みにしない**。
- **世界を変えた10回の平均と誤差**で比べる。観測テストの差が、本当の良さの差とは限らない（バイアスモデル対平均評価順）。

次の Phase2 では、**近傍法**（似たユーザー・似た商品から薦める）を作り、人気順と、**観測テストと本当の好みの両方**で、比べます。